In [1]:
import sys
from pathlib import Path

import numpy as np
import pyomo.environ as pyo

# Obtiene la ruta de la carpeta raíz (un nivel arriba de notebooks)
root_path = Path.cwd().parent
sys.path.append(str(root_path))

from src.models_deterministic import ServicioTren, TipoTren
from src.models_deterministic import ModeloTrenesUnitariosDeterminista, DatosRedFerroviaria

%load_ext autoreload
%autoreload 2

In [ ]:
# FUNCIONES AUXILIARES PARA GENERAR DATOS DE EJEMPLO

def generar_datos_ejemplo_pequeno() -> DatosRedFerroviaria:
    """
    Genera un conjunto de datos de ejemplo para probar el modelo.
    Este ejemplo simula una red pequeña con 2 estaciones, 2 trenes y 2 servicios.
    """

    # H: int  Horizonte de planificación (días de salida)
    # K: int  Número de clases de servicio
    # S: int  Número de estaciones
    # N: int  Número de servicios programados
    # R: int  Número de trenes físicos disponibles
    # O: int  Número de opciones de precio
    # SD: int  Clases de estacionalidad
    # TD: int  Clases de tiempo-anticipación

    #  Dimensiones 
    H, K, S, N, R, O, SD, TD = 3, 2, 2, 2, 2, 3, 2,

    #  Tipos de tren 
    tipos_tren = [
        TipoTren(
            "Premium",
            capacidad=40,
            costo_fijo=8500,
            costo_variable_base=75,
            multiplicador_clase=[1.5, 1.0, 0.7],
        ),
        TipoTren(
            "LowCost",
            capacidad=60,
            costo_fijo=5000,
            costo_variable_base=45,
            multiplicador_clase=[1.5, 1.0, 0.7],
        ),
    ]

    # Tren 1: Premium, Tren 2: LowCost
    tipo_tren_por_r = [0, 1]

    #  Servicios 
    servicios = [
        ServicioTren(id=1, origen=1, destino=2, duracion=1, horarios=[1, 2, 3]),
        ServicioTren(id=2, origen=2, destino=1, duracion=1, horarios=[1, 2, 3]),
    ]

    #  Programación 
    PL = np.ones((N, H))  # Todos los servicios operan todos los días

    #  Matriz de rutas (opcional) 
    TR = np.zeros((N, S, S))
    for idx, srv in enumerate(servicios):
        TR[idx, srv.origen - 1, srv.destino - 1] = 1

    #  Posiciones iniciales 
    pos_inicial = np.zeros((R, S))
    pos_inicial[0, 0] = 1  # Tren 1 en estación 1
    pos_inicial[1, 1] = 1  # Tren 2 en estación 2

    #  Precios 
    precios = np.zeros((K, N, O))
    for k in range(K):
        for n in range(N):
            base = 100 if n == 0 else 80
            mult_clase = [1.5, 1.0][k]
            for o in range(O):
                mult_precio = [0.8, 1.0, 1.2][o]
                precios[k, n, o] = base * mult_clase * mult_precio

    #  Parámetros económicos 
    CC, OC, AR = 200.0, 65000.0, 0.0

    #  Estacionalidad 
    SE = np.ones((H, N), dtype=int)  # Siempre temporada media

    #  Demanda determinista 
    DE = np.ones((SD, TD, K, N, O)) * 30.0
    DE[:, :, :, :, 0] = 45.0  # Precio bajo: más demanda
    DE[:, :, :, :, 1] = 30.0  # Precio medio: demanda media
    DE[:, :, :, :, 2] = 20.0 if O > 2 else 30.0  # Precio alto: menos demanda

    #  Ventas previas 
    RW = np.zeros((H, K, N))
    RS = np.zeros((H, K, N))

    return DatosRedFerroviaria(
        H=H,
        K=K,
        S=S,
        N=N,
        R=R,
        O=O,
        SD=SD,
        TD=TD,
        tipo_tren_por_r=tipo_tren_por_r,
        tipos_tren=tipos_tren,
        servicios=servicios,
        PL=PL,
        TR=TR,
        pos_inicial=pos_inicial,
        precios=precios,
        CC=CC,
        OC=OC,
        AR=AR,
        SE=SE,
        DE=DE,
        RW=RW,
        RS=RS,
        MAX_COUPLE=1,
        EPSILON=0.01,
        BIG_M=1e6,
    )

In [3]:
print("=" * 70)
print("MODELO DETERMINISTA DE TRENES UNITARIOS")
print("=" * 70)

# Generar datos de ejemplo
print("\n📊 Generando datos de ejemplo...")
datos = generar_datos_ejemplo_pequeno()

print(f"  - Horizonte: {datos.H} días")
print(f"  - Servicios: {datos.N}")
print(
    f"  - Trenes: {datos.R} (Premium: {datos.tipo_tren_por_r.count(0)}, LowCost: {datos.tipo_tren_por_r.count(1)})"
)
print(f"  - Estaciones: {datos.S}")
print(f"  - Clases: {datos.K}")

# Crear modelo
print("\n🔧 Construyendo modelo Pyomo...")
modelo = ModeloTrenesUnitariosDeterminista(datos)

# Mostrar estadísticas
num_vars = len(list(modelo.modelo.component_objects(pyo.Var, active=True)))
num_constr = len(list(modelo.modelo.component_objects(pyo.Constraint, active=True)))
print(f"  - Variables: {num_vars} grupos")
print(f"  - Restricciones: {num_constr} grupos")

# Resolver
print("\n🚀 Resolviendo con Gurobi...")
resultado = modelo.resolver(
    solver_name="mindtpy", time_limit=60, gap=0.01, verbose=True
)

print(resultado)

# print(f"\n📈 Estado de la solución: {resultado['status']}")
# print(f"⏱️  Tiempo: {resultado['tiempo']:.2f} segundos")

# if resultado["gap"] is not None:
#     print(f"📊 Gap de optimalidad: {resultado['gap']:.4%}")

# # Mostrar resultados
# modelo.imprimir_resumen()


MODELO DETERMINISTA DE TRENES UNITARIOS

📊 Generando datos de ejemplo...
  - Horizonte: 3 días
  - Servicios: 2
  - Trenes: 2 (Premium: 1, LowCost: 1)
  - Estaciones: 2
  - Clases: 2

🔧 Construyendo modelo Pyomo...
  - Variables: 8 grupos
  - Restricciones: 22 grupos

🚀 Resolviendo con Gurobi...


Starting MindtPy version 1.0.0 using OA algorithm
iteration_limit: 50
stalling_limit: 15
time_limit: 300
strategy: OA
add_regularization: None
call_after_main_solve: <pyomo.contrib.gdpopt.util._DoNothing object at 0x7f591f00ef30>
call_before_subproblem_solve: <pyomo.contrib.gdpopt.util._DoNothing object at 0x7f591f00ef60>
call_after_subproblem_solve: <pyomo.contrib.gdpopt.util._DoNothing object at 0x7f591f00ef90>
call_after_subproblem_feasible: <pyomo.contrib.gdpopt.util._DoNothing object at 0x7f591f00efc0>
tee: true
logger: <Logger pyomo.contrib.mindtpy (INFO)>
logging_level: 20
integer_to_binary: false
add_no_good_cuts: false
use_tabu_list: false
single_tree: false
solution_pool: false
num_solution_iteration: 5
cycling_check: true
feasibility_norm: L_infinity
differentiate_mode: reverse_symbolic
use_mcpp: false
calculate_dual_at_solution: false
use_fbbt: false
use_dual_bound: true
partition_obj_nonlinear_terms: true
quadratic_strategy: 0
move_objective: false
add_cuts_at_incumbent: f

deprecated. Either specify deactivated Blocks as targets to activate them if
transforming them is the desired behavior.  (deprecated in 6.9.3) (called from
/home/mikdevio/.pyenv/versions/.train_revenue_management/lib/python3.12/site-
packages/pyomo/core/base/transformation.py:75)


         -       Relaxed NLP            -65000           -inf         -65000      nan%      0.58
         1              MILP            -65000           -inf         -65000      nan%      0.66
*        1         Fixed NLP            -65000         -65000         -65000    -0.00%      0.76
MindtPy exiting on bound convergence. Absolute gap: -3.6811688914895058e-06 <= absolute tolerance: 0.0001 

 Primal integral          :    0.0000 
 Dual integral            :    0.0001 
 Primal-dual gap integral :    0.0001 


{'status': <TerminationCondition.optimal: 'optimal'>, 'tiempo': None, 'gap': None}


In [4]:
modelo.modelo.pprint()

7 RangeSet Declarations
    CLASES : Dimen=1, Size=2, Bounds=(1, 2)
        Key  : Finite : Members
        None :   True :   [1:2]
    DIAS : Dimen=1, Size=3, Bounds=(1, 3)
        Key  : Finite : Members
        None :   True :   [1:3]
    ESTACIONES : Dimen=1, Size=2, Bounds=(1, 2)
        Key  : Finite : Members
        None :   True :   [1:2]
    OPCIONES : Dimen=1, Size=3, Bounds=(1, 3)
        Key  : Finite : Members
        None :   True :   [1:3]
    SERVICIOS : Dimen=1, Size=2, Bounds=(1, 2)
        Key  : Finite : Members
        None :   True :   [1:2]
    TIEMPO : Dimen=1, Size=3, Bounds=(1, 3)
        Key  : Finite : Members
        None :   True :   [1:3]
    TRENES : Dimen=1, Size=2, Bounds=(1, 2)
        Key  : Finite : Members
        None :   True :   [1:2]

17 Param Declarations
    AR : Size=1, Index=None, Domain=Any, Default=None, Mutable=False
        Key  : Value
        None :   0.0
    BIG_M : Size=1, Index=None, Domain=Any, Default=None, Mutable=False
       

In [12]:
modelo.modelo.display()

Model unknown

  Variables:
    pos : Size=12, Index=TRENES*ESTACIONES*DIAS
        Key       : Lower : Value : Upper : Fixed : Stale : Domain
        (1, 1, 1) :     0 :   1.0 :     1 : False : False : Binary
        (1, 1, 2) :     0 :   1.0 :     1 : False : False : Binary
        (1, 1, 3) :     0 :   1.0 :     1 : False : False : Binary
        (1, 2, 1) :     0 :   0.0 :     1 : False : False : Binary
        (1, 2, 2) :     0 :   0.0 :     1 : False : False : Binary
        (1, 2, 3) :     0 :   0.0 :     1 : False : False : Binary
        (2, 1, 1) :     0 :   0.0 :     1 : False : False : Binary
        (2, 1, 2) :     0 :   0.0 :     1 : False : False : Binary
        (2, 1, 3) :     0 :   0.0 :     1 : False : False : Binary
        (2, 2, 1) :     0 :   1.0 :     1 : False : False : Binary
        (2, 2, 2) :     0 :   1.0 :     1 : False : False : Binary
        (2, 2, 3) :     0 :   1.0 :     1 : False : False : Binary
    u : Size=12, Index=TRENES*SERVICIOS*DIAS
        